### Inspect the gene names column from both reference and spatial adatas to find correct common column

In [3]:
# From your reference
print("Reference gene names (first 10):", gbmap.var_names[:10].tolist())

# From a spatial sample
sample_adata = sc.read_h5ad(adata_files[0])
print("Spatial gene names (first 10):", sample_adata.var_names[:10].tolist())


Reference gene names (first 10): ['ENSG00000069431', 'ENSG00000107796', 'ENSG00000151388', 'ENSG00000145536', 'ENSG00000156140', 'ENSG00000120907', 'ENSG00000170214', 'ENSG00000204305', 'ENSG00000204472', 'ENSG00000171094']
Spatial gene names (first 10): ['ABCC9', 'ACTA2', 'ADAMTS12', 'ADAMTS16', 'ADAMTS3', 'ADRA1A', 'ADRA1B', 'AGER', 'AIF1', 'ALK']


# Preparing Files for RCTD Analysis

## Overview

To run RCTD (Robust Cell Type Decomposition) on our spatial transcriptomics data, we need:
- Spatial counts matrices (spots × genes) for each sample
- Coordinates for each spatial spot
- Reference (single-cell) counts matrix (cells × genes) with matching gene names
- Reference cell type labels


### Import libraries and set file paths

In [11]:
import scanpy as sc
import pandas as pd
import glob
import os
import mygene
import numpy as np

# --- Set your paths ---
spatial_folder = '/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/out_july2025_xenium/script01_adatas/'
counts_out_folder = '/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/rctd_data_prep/spatial_counts_for_rctd/'
coords_out_folder = '/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/rctd_data_prep/spatial_coords_for_rctd/'
reference_out_folder = '/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/rctd_data_prep/reference_for_rctd/'
reference_file = '/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/out_feb2025_xenium/script02a_gbmap_reduced/gbmap_reduced.h5ad'

os.makedirs(counts_out_folder, exist_ok=True)
os.makedirs(coords_out_folder, exist_ok=True)

# --- Find all .h5ad files ---
adata_files = glob.glob(os.path.join(spatial_folder, '*.h5ad'))
print(f"Found {len(adata_files)} spatial samples.")


Found 40 spatial samples.


### 1. Reference Gene Mapping

Our single-cell reference data used **Ensembl gene IDs** while spatial samples use **gene symbols**.
- We mapped Ensembl IDs to gene symbols using the `mygene` Python package.
- After mapping, we ensured unique gene symbols and intersected with spatial genes.


In [12]:
# --- Load reference AnnData and map Ensembl IDs to gene symbols ---
gbmap = sc.read_h5ad(reference_file)
ensembl_ids = gbmap.var_names.tolist()

print("Mapping Ensembl IDs to gene symbols using mygene...")
mg = mygene.MyGeneInfo()
result = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')

ensembl2symbol = {}
for r in result:
    if 'symbol' in r and r['symbol'] not in ensembl2symbol.values():
        ensembl2symbol[r['query']] = r['symbol']

# Only keep genes with unique symbols
ensembl_keep = [g for g in gbmap.var_names if g in ensembl2symbol]
symbols = [ensembl2symbol[g] for g in ensembl_keep]
gbmap_symbol = gbmap[:, ensembl_keep].copy()
gbmap_symbol.var_names = symbols

# Drop duplicate gene symbols in the reference (keep first occurrence)
_, idx = np.unique(gbmap_symbol.var_names, return_index=True)
gbmap_symbol = gbmap_symbol[:, idx]
gbmap_symbol.var_names = gbmap_symbol.var_names[idx]


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Mapping Ensembl IDs to gene symbols using mygene...


### 2. Export Reference Data

- Exported a filtered reference counts matrix (`sc_reference_counts.csv`) using only the shared genes.
- Exported the corresponding cell type labels (`sc_reference_celltypes.csv`).


In [13]:
# --- Use first sample to intersect shared genes (now symbols) ---
sample_adata = sc.read_h5ad(adata_files[0])
shared_genes = gbmap_symbol.var_names.intersection(sample_adata.var_names)
print(f"Shared genes between reference and first sample: {len(shared_genes)}")

gbmap_sub = gbmap_symbol[:, shared_genes].copy()

# --- Export reference (run ONCE) ---
ref_counts = pd.DataFrame(
    gbmap_sub.X.toarray() if hasattr(gbmap_sub.X, "toarray") else gbmap_sub.X,
    index=gbmap_sub.obs_names,
    columns=gbmap_sub.var_names
)
print(f"Reference counts shape: {ref_counts.shape}")
ref_counts.to_csv("sc_reference_counts.csv")
celltype_col = 'cell_type'  # <-- edit if your label is named differently
gbmap_sub.obs[celltype_col].to_csv("sc_reference_celltypes.csv", header=True)


Shared genes between reference and first sample: 356


Reference counts shape: (1135677, 356)


### 3. Export Spatial Data

For each spatial sample:
- Exported the spatial counts matrix (spots × genes) containing only the shared, harmonized genes.
- Exported the spatial coordinates (x, y) for each spot.

Due to a few large files we had to go with separate batch analysis while looping through the h5ad files of anndata!


In [4]:
# --- Loop over the first 10 spatial samples ---
for file in adata_files[20:30]:
    basename = os.path.splitext(os.path.basename(file))[0]  # correct extraction
    print(f"\nProcessing {basename} ...")
    xe = sc.read_h5ad(file)
    
    # Intersect with reference (gene symbols)
    genes_this_sample = gbmap_symbol.var_names.intersection(xe.var_names)
    print(f"Shared genes in this sample: {len(genes_this_sample)}")
    
    xe_sub = xe[:, genes_this_sample].copy()
    ref_sub = gbmap_symbol[:, genes_this_sample].copy()

    # Export spatial counts
    df_counts = pd.DataFrame(
        xe_sub.X.toarray() if hasattr(xe_sub.X, "toarray") else xe_sub.X,
        index=xe_sub.obs_names,
        columns=xe_sub.var_names
    )
    print(f"Counts DataFrame shape: {df_counts.shape}")
    print(df_counts.iloc[:2, :2])  # Show top-left for sanity

    out_counts = os.path.join(counts_out_folder, f"{basename}_spatial_counts.csv")
    df_counts.to_csv(out_counts)

    # Export coordinates if available
    if "spatial" in xe_sub.obsm:
        df_coords = pd.DataFrame(
            xe_sub.obsm["spatial"],
            index=xe_sub.obs_names,
            columns=["x", "y"]
        )
        print(f"Coords DataFrame shape: {df_coords.shape}")
        print(df_coords.head())
        out_coords = os.path.join(coords_out_folder, f"{basename}_spatial_coords.csv")
        df_coords.to_csv(out_coords)
    else:
        print(f"No spatial coordinates found for {basename}!")

    print(f"Exported: {out_counts} (and coords if available)")

print("All files exported and verified. Ready for R/RCTD step. ")


Processing adata_Control_2 ...
Shared genes in this sample: 356


Counts DataFrame shape: (8108, 356)
            ABCC9     ACTA2
ahemannf-1    0.0  0.424883
fnfngiib-1    0.0  0.000000
Coords DataFrame shape: (8108, 2)
                      x            y
ahemannf-1  3138.109863  1703.594116
fnfngiib-1  3140.430420  1692.118286
gflloajj-1  3143.970703  1686.237793
gblllakh-1  3149.562256  1700.882568
agpeliih-1  3148.073975  1711.156982
Exported: /home/ext_sana_noor_astraeabio_com/ext_hd/chiba/rctd_data_prep/spatial_counts_for_rctd/adata_Control_2_spatial_counts.csv (and coords if available)

Processing adata_Oligo5_Core ...
Shared genes in this sample: 351
Counts DataFrame shape: (2833, 351)
            ABCC9  ACTA2
icahdmkl-1    0.0    0.0
ienoaiak-1    0.0    0.0
Coords DataFrame shape: (2833, 2)
                      x            y
icahdmkl-1  1652.052612   722.928467
ienoaiak-1  1675.636353   821.436951
ibfahlia-1   843.314392  1651.728027
fmdnoccn-1   822.062988  1657.795898
fmdnplbg-1   835.286316  1662.239014
Exported: /home/ext_sana_noor_as